In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# 1. Load the dataset and inspect overview details
# (Update 'HR_comma_sep.csv' if your filename differs)
df = pd.read_csv('HR_comma_sep.csv')

print("=== 1. Dataset Overview ==ppage ===")
print("First 5 rows:")
display(df.head())  # Use print(df.head()) if running in a standard terminal/script

print("\nDataset Shape (Rows, Columns):", df.shape)

print("\nMissing-Value Counts:")
print(df.isnull().sum())

print("\nTarget Class Counts ('left'):")
print(df['left'].value_counts())

# 2. Set 'left' as the target and use remaining columns as features
X = df.drop(columns=['left'])
y = df['left']

# 3. Split the dataset into 70% training and 30% testing, using random_state=25 and stratify=y
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=25, stratify=y
)

# 4. Setup ColumnTransformer for OneHotEncoder and keep numerical features unchanged
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_features = X.select_dtypes(include=['number']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ],
    remainder='passthrough'  # Keeps numerical features unchanged
)

# 5. Fit the transformer on the training data only, then transform both training and test data
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print("\nData preparation complete!")
print("X_train_transformed shape:", X_train_transformed.shape)
print("X_test_transformed shape:", X_test_transformed.shape)

=== 1. Dataset Overview ==ppage ===
First 5 rows:


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low



Dataset Shape (Rows, Columns): (14995, 10)

Missing-Value Counts:
satisfaction_level       0
last_evaluation          0
number_project           0
average_montly_hours     0
time_spend_company       0
Work_accident            0
left                     0
promotion_last_5years    0
Department               0
salary                   0
dtype: int64

Target Class Counts ('left'):
left
0    11428
1     3567
Name: count, dtype: int64

Data preparation complete!
X_train_transformed shape: (10496, 18)
X_test_transformed shape: (4499, 18)


C:\Users\HP\AppData\Local\Temp\ipykernel_7764\2025036047.py:32: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()


In [2]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# 6. Create the five base classifiers with specified settings
TREE1 = DecisionTreeClassifier(random_state=25)
TREE2 = DecisionTreeClassifier(max_depth=3, random_state=25)
KNN1 = KNeighborsClassifier(n_neighbors=5)
KNN2 = KNeighborsClassifier(n_neighbors=3)
NB = GaussianNB()

# Optional: Store them in a dictionary for easy looping and evaluation in subsequent steps
base_classifiers = {
    'TREE1': TREE1,
    'TREE2': TREE2,
    'KNN1': KNN1,
    'KNN2': KNN2,
    'NB': NB
}

print("All 5 base classifiers successfully initialized!")

All 5 base classifiers successfully initialized!


In [3]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, classification_report

# 7. Combine the five classifiers in a VotingClassifier using voting='hard' (equal weights by default)
estimators = [
    ('TREE1', TREE1),
    ('TREE2', TREE2),
    ('KNN1', KNN1),
    ('KNN2', KNN2),
    ('NB', NB)
]

hard_voting_clf = VotingClassifier(
    estimators=estimators,
    voting='hard'
)

# 8. Train the ensemble on the transformed training data
hard_voting_clf.fit(X_train_transformed, y_train)

# 9. Predict test labels and display accuracy and classification report
y_pred_hard = hard_voting_clf.predict(X_test_transformed)
hard_accuracy = accuracy_score(y_test, y_pred_hard)

print(f"=== Hard Voting Ensemble Test Accuracy: {hard_accuracy:.4f} ==-\n")
print("=== Classification Report (Hard Voting) ===")
print(classification_report(y_test, y_pred_hard))

=== Hard Voting Ensemble Test Accuracy: 0.9627 ==-

=== Classification Report (Hard Voting) ===
              precision    recall  f1-score   support

           0       0.99      0.97      0.98      3429
           1       0.90      0.95      0.92      1070

    accuracy                           0.96      4499
   macro avg       0.94      0.96      0.95      4499
weighted avg       0.96      0.96      0.96      4499



In [4]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, classification_report, log_loss

# 11. Create a VotingClassifier with voting='soft' and specified weights
estimators = [
    ('TREE1', TREE1),
    ('TREE2', TREE2),
    ('KNN1', KNN1),
    ('KNN2', KNN2),
    ('NB', NB)
]

# Specified weights: TREE1 (4), TREE2 (2.5), KNN1 (1), KNN2 (1), NB (1)
weights = [4, 2.5, 1, 1, 1]

soft_voting_clf = VotingClassifier(
    estimators=estimators,
    voting='soft',
    weights=weights
)

# 12. Train the ensemble on the transformed training data
soft_voting_clf.fit(X_train_transformed, y_train)

# 13. Predict test labels and display accuracy and classification report
y_pred_soft = soft_voting_clf.predict(X_test_transformed)
soft_accuracy = accuracy_score(y_test, y_pred_soft)

print(f"=== Weighted Soft Voting Ensemble Test Accuracy: {soft_accuracy:.4f} ===\n")
print("=== Classification Report (Soft Voting) ===")
print(classification_report(y_test, y_pred_soft))

# 14. Predict test-set class probabilities and calculate Log Loss
y_prob_soft = soft_voting_clf.predict_proba(X_test_transformed)
soft_log_loss = log_loss(y_test, y_prob_soft)

print(f"\nWeighted Soft Voting Test Log Loss: {soft_log_loss:.4f}")

=== Weighted Soft Voting Ensemble Test Accuracy: 0.9842 ===

=== Classification Report (Soft Voting) ===
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3429
           1       0.96      0.97      0.97      1070

    accuracy                           0.98      4499
   macro avg       0.98      0.98      0.98      4499
weighted avg       0.98      0.98      0.98      4499


Weighted Soft Voting Test Log Loss: 0.0968


In [5]:
from sklearn.metrics import precision_recall_fscore_support


# Function to extract metrics for Class 1 (Left = 1) from classification reports / metrics
def get_class_1_metrics(y_true, y_pred, method_name, accuracy):
    # precision, recall, f1, support for class 1 (index 1)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average=None)
    return {
        'Voting Method': method_name,
        'Accuracy': round(accuracy, 4),
        'Precision for Class 1': round(precision[1], 4),
        'Recall for Class 1': round(recall[1], 4),
        'F1-score for Class 1': round(f1[1], 4)
    }

# Gather metrics for Hard Voting and Soft Voting
hard_metrics = get_class_1_metrics(y_test, y_pred_hard, 'Hard Voting', hard_accuracy)
soft_metrics = get_class_1_metrics(y_test, y_pred_soft, 'Weighted Soft Voting', soft_accuracy)

# 16. Create the comparison DataFrame
comparison_df = pd.DataFrame([hard_metrics, soft_metrics])

print("=== Part E: Voting Method Comparison Table ===")
print(comparison_df.to_string(index=False))

=== Part E: Voting Method Comparison Table ===
       Voting Method  Accuracy  Precision for Class 1  Recall for Class 1  F1-score for Class 1
         Hard Voting    0.9627                 0.8963              0.9533                0.9239
Weighted Soft Voting    0.9842                 0.9638              0.9701                0.9669
